## Data skew — spotting and fixing it

**Symptom in the UI:** stage duration 25 minutes; 199 of 200 tasks finished in 30 seconds; one task still running at 24 minutes. Task summary: median shuffle read 400 MB, **max 5 GB**.

**Diagnosis:** the join / aggregation key is unbalanced. One `customer_id` — a corporate customer with 50 million transactions — has 100× the rows of the median. After the shuffle, all those rows land on one executor task, and that one task holds up the whole stage.

**Three remedies, in the order the exam expects:**

1. **Enable AQE skew-join** — the recommended answer for almost every exam question. AQE detects oversized partitions at runtime, splits them into sub-partitions, and joins each independently. **Zero code change.**

    ```python
    spark.conf.set("spark.sql.adaptive.enabled", "true")
    spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
    ```

2. **Broadcast** the small side if it fits in driver memory — eliminates the shuffle entirely (module 04).

3. **Salt the join key** when AQE isn't enough — add a random `0..N-1` to the key, join, then aggregate the salt away. Heavier surgery; keep it for when AQE has been tried and isn't sufficient.

**The exam answer ~99% of the time is option 1 — enable AQE skew-join.** The official sample question names exactly this. Reach for salt only when a question explicitly rules AQE out.
